# Real BEELINE benchmark GRN: load & visualize (GSD)

This notebook is the BEELINE-data counterpart to `synthetic_grn_demo.ipynb`. Instead of *simulating* data from an iqcell `GRNSpec`, it downloads a **real BEELINE/BoolODE benchmark dataset** and runs the same load → visualize → analyze → feed-the-pipeline steps on it.

**Dataset: GSD (Gonadal Sex Determination).** A curated Boolean model from the [BEELINE](https://github.com/Murali-group/Beeline) `inputs/example/GSD` benchmark (Pratapa et al., *Nature Methods* 2020). BoolODE simulated the model into scRNA-seq-like expression along pseudotime; the ground-truth signed network is the biology being recovered.

**Biology modelled:** the male (SRY → SOX9 → FGF9 feed-forward) vs. female (WNT4/RSPO1 → CTNNB1 → FOXL2) fate decision, resolved by mutual antagonism (SOX9 ⊣ FOXL2 and back). The two pseudotime branches correspond to the two resolved fates.

Because the topology is known, this is a ground-truth benchmark for the iqcell inference modules (binarization → gene hierarchy → logic engine) — exactly like the synthetic notebook, but on externally-generated data.

## Setup

In [ ]:
import os, sys, io, urllib.request
# Make the repo root importable when running from examples/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, os.path.abspath(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

np.random.seed(0)
print('imports ok')

## 0. Download the BEELINE GSD dataset

We pull the three canonical BEELINE files straight from the Beeline repo (`inputs/example/GSD/`): the dataset-level `GroundTruthNetwork.csv` and the `ex1/` run's `ExpressionData.csv` + `PseudoTime.csv`. Files are cached under `data/beeline_gsd/` so the notebook is offline-friendly after the first run.

In [ ]:
BASE = 'https://raw.githubusercontent.com/Murali-group/Beeline/master/inputs/example/GSD'
DATA_DIR = os.path.abspath('data/beeline_gsd')
FILES = {
    'network':    ('GroundTruthNetwork.csv', 'GroundTruthNetwork.csv'),
    'expression': ('ex1/ExpressionData.csv',  'ExpressionData.csv'),
    'pseudotime': ('ex1/PseudoTime.csv',       'PseudoTime.csv'),
}
os.makedirs(DATA_DIR, exist_ok=True)

paths = {}
for key, (remote, local) in FILES.items():
    dest = os.path.join(DATA_DIR, local)
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f'{BASE}/{remote}', dest)
        print('downloaded', local)
    else:
        print('cached    ', local)
    paths[key] = dest

## 1. Load the ground-truth GRN

BEELINE ground-truth networks are edge lists: `Gene1,Gene2,Type` where `Type` is `+` (activation) or `-` (repression). This is the analogue of defining a `GRNSpec` in the synthetic notebook — except here the topology is *given*.

In [ ]:
net = pd.read_csv(paths['network'])
net = net.drop_duplicates().reset_index(drop=True)  # BoolODE lists some edges twice
genes = sorted(set(net['Gene1']) | set(net['Gene2']))
print(f'{len(genes)} genes, {len(net)} unique signed edges')
print('genes:', genes)
net.head(10)

### Signed adjacency matrix
Rows = regulator, columns = target. `+1` = activation, `-1` = repression, `0` = no edge. (Self-loops like `SOX9 -> SOX9` show on the diagonal.)

In [ ]:
sign_map = {'+': 1, '-': -1}
adj = pd.DataFrame(0, index=genes, columns=genes, dtype=int)
for _, row in net.iterrows():
    adj.loc[row['Gene1'], row['Gene2']] = sign_map[str(row['Type']).strip()]
adj

## 2. Visualize the GRN graph
Green arrows = activation, red bracket-heads = repression. Node color encodes out-degree (how many genes each one regulates) — the master regulators pop out. This mirrors the circular-layout plot in the synthetic notebook, using a spring layout better suited to a denser 19-gene network.

In [ ]:
G = nx.DiGraph()
G.add_nodes_from(genes)
for _, r in net.iterrows():
    G.add_edge(r['Gene1'], r['Gene2'], sign=sign_map[str(r['Type']).strip()])

pos = nx.spring_layout(G, seed=3, k=1.1)
out_deg = dict(G.out_degree())
node_colors = [out_deg[g] for g in G.nodes]

act = [(u, v) for u, v, s in G.edges(data='sign') if s > 0]
rep = [(u, v) for u, v, s in G.edges(data='sign') if s < 0]

fig, ax = plt.subplots(figsize=(11, 9))
nodes = nx.draw_networkx_nodes(G, pos, node_color=node_colors, cmap='YlOrRd',
                               node_size=1500, edgecolors='black', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=act, edge_color='#2a9d8f', width=1.6,
                       arrowsize=14, node_size=1500,
                       connectionstyle='arc3,rad=0.12', ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=rep, edge_color='#e63946', width=1.6,
                       arrowsize=14, arrowstyle='-[', node_size=1500,
                       connectionstyle='arc3,rad=0.12', ax=ax)
fig.colorbar(nodes, ax=ax, label='out-degree (# targets regulated)', shrink=0.7)
ax.set_title('GSD ground-truth GRN (green=activation, red=repression)')
ax.set_axis_off(); plt.tight_layout(); plt.show()

## 3. Load the scRNA-seq-like expression data

BoolODE produced a genes × cells matrix along two pseudotime trajectories (the two resolved fates). This is the analogue of `.simulate()` in the synthetic notebook — the data already exists, we just load it.

In [ ]:
expr = pd.read_csv(paths['expression'], index_col=0)   # genes x cells
expr = expr.loc[genes]                                   # align to network gene order
ptime = pd.read_csv(paths['pseudotime'], index_col=0)    # cells x branches
print('expression (genes x cells):', expr.shape)
print('pseudotime branches       :', list(ptime.columns))
print('cells per branch (non-NA) :')
print(ptime.notna().sum())
expr.iloc[:5, :4]

### Expression along pseudotime (per branch)
Each cell belongs to one of two trajectories. We collapse the two pseudotime columns into one ordering per cell and plot a few key fate genes. Watch the **SOX9 (male) vs. FOXL2 (female)** antagonism resolve along pseudotime.

In [ ]:
# A single per-cell pseudotime = whichever branch column is non-NA.
pt = ptime.bfill(axis=1).iloc[:, 0].astype(float)
branch = ptime.notna().idxmax(axis=1)      # which branch each cell is on

watch = [g for g in ['SRY', 'SOX9', 'FGF9', 'FOXL2', 'WNT4', 'CTNNB1'] if g in expr.index]
color = plt.cm.tab10(np.linspace(0, 1, len(watch)))

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, br in zip(axes, ptime.columns):
    cells_br = branch[branch == br].index
    order = pt[cells_br].sort_values().index
    for g, c in zip(watch, color):
        ax.plot(pt[order].values, expr.loc[g, order].values, label=g, lw=1.6, color=c)
    ax.set_title(f'Fate branch: {br}'); ax.set_xlabel('pseudotime')
axes[0].set_ylabel('expression'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Expression heatmap (genes × cells)
Cells ordered by pseudotime along the x-axis (both branches concatenated).

In [ ]:
order_all = pt.sort_values().index
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(expr[order_all].values, aspect='auto', cmap='viridis',
               interpolation='nearest')
ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes, fontsize=7)
ax.set_xlabel('cell (ordered by pseudotime)')
ax.set_title('GSD expression heatmap')
fig.colorbar(im, ax=ax, label='expression')
plt.tight_layout(); plt.show()

## 4. Binarize for the iqcell pipeline

The synthetic notebook produced binarized ON/OFF states via the generator. Here we run the *same* iqcell binarizer used downstream on real BEELINE data: build an `AnnData` (cells × genes) and discretize per gene. This is the entry point to the binarization → gene-hierarchy → logic-engine pipeline.

In [ ]:
import anndata as ad
from iqcell.binarization.mean import Mean

X = expr.T.values.astype(float)                       # cells x genes
adata = ad.AnnData(X=X.copy())
adata.var_names = list(expr.index)
adata.obs_names = list(expr.columns)
adata.obs['pseudotime'] = pt.reindex(adata.obs_names).values
adata.raw = adata                                     # continuous kept in raw.X

binarized = Mean().discretize(adata)                  # per-gene threshold = column mean
print(adata)
print('binarized X unique values:', np.unique(binarized.X))

### Binarized ON/OFF states
Cells ordered by pseudotime. The male/female fate genes switch ON/OFF in opposition — the discrete signal the iqcell logic engine consumes.

In [ ]:
Xb = np.asarray(binarized.X)
order_idx = [adata.obs_names.get_loc(c) for c in order_all]
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(Xb[order_idx].T, aspect='auto', cmap='Greys',
               interpolation='nearest', vmin=0, vmax=1)
ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes, fontsize=7)
ax.set_xlabel('cell (ordered by pseudotime)')
ax.set_title('Binarized (ON/OFF) states — real BEELINE GSD data')
fig.colorbar(im, ax=ax, label='state', ticks=[0, 1])
plt.tight_layout(); plt.show()

## 5. Ground-truth for benchmarking inference

The real payoff of a BEELINE dataset: the known signed network is the target any inference algorithm must recover. `iqcell.beeline.score_ranking` scores a ranked edge list against it (AUPRC/AUROC over all directed gene pairs). Below we show the scorer on an **oracle** ranking (the true edges) vs. a **random** one — the same sanity check used in the BEELINE benchmark notebook.

In [ ]:
from iqcell.beeline import score_ranking

ground_truth = [(r['Gene1'], r['Gene2'], sign_map[str(r['Type']).strip()])
                for _, r in net.iterrows()]

oracle = pd.DataFrame([(g1, g2, 1.0) for g1, g2, _ in ground_truth],
                      columns=['Gene1', 'Gene2', 'EdgeWeight'])
rng = np.random.default_rng(0)
all_pairs = [(a, b) for a in genes for b in genes if a != b]
random_rank = pd.DataFrame([(a, b, float(rng.random())) for a, b in all_pairs],
                           columns=['Gene1', 'Gene2', 'EdgeWeight'])

print('oracle ranking:', score_ranking(oracle, ground_truth))
print('random ranking:', score_ranking(random_rank, ground_truth))

## Summary

- Loaded a **real BEELINE/BoolODE benchmark dataset** (GSD, 19 genes × 2000 cells, 2 fate branches) instead of simulating one.
- Parsed the ground-truth signed network into an adjacency matrix and drew the regulatory graph (activation vs. repression, master regulators by out-degree).
- Loaded the scRNA-seq-like expression, plotted per-branch trajectories and a pseudotime-ordered heatmap.
- Binarized it through the **same iqcell binarizer** the downstream pipeline uses, and visualized the ON/OFF states.
- Used the known topology as a ground-truth benchmark and scored oracle vs. random rankings with `iqcell.beeline.score_ranking`.

This is the external-data analogue of `synthetic_grn_demo.ipynb`: same workflow, real BEELINE data. To run actual inference algorithms on it, point `BeelineRunner` at a bootstrapped BEELINE checkout — see `docs/beeline.md`.